# **PR0302: Lectura avanzada de datos de archivos**

## **Objetivo de la Práctica**

Esta práctica es una revisión de la anterior donde aumentamos el nivel de dificultad ya que los archivos de origen tienen sus peculiaridades.

Nuevamente tienes que leer datos de tres fuentes diferentes (CSV, Excel y JSON) y volcar el resultado a un archivo CSV.

## **1. Norte** (``ventas_norte_v2.csv``):
- Formato CSV delimitado por punto y coma (``;``).
- El campo ``Direccion_Envio`` contiene comas reales (ej: “Calle Mayor, 12”). Debes asegurarte de que Pandas no rompa las filas al leerlo.
- El campo ``Total_Factura`` viene como texto con símbolos de moneda (ej: “$1,200.50”). Deberás limpiarlo y convertirlo a ``float``.

In [9]:
import pandas as pd

def limpiar_moneda(x):
    return float(x.replace("$",""))

df_norte = pd.read_csv(
    "./ventas_norte_v2.csv",
    delimiter=";",
    header=0,
    parse_dates=["Fecha_Transaccion"],
    converters={"Total_Factura": limpiar_moneda}
)

In [8]:
print(df_norte.head(20))

   ID_Pedido   Fecha_Transaccion Cliente_Nombre            Direccion_Envio  \
0     N-1000 2023-05-10 03:00:00      Usuario_0       Gran Vía, 49, Piso 4   
1     N-1001 2023-06-18 09:00:00      Usuario_1   Av. Libertad, 30, Piso 4   
2     N-1002 2023-03-10 05:00:00      Usuario_2   Av. Libertad, 98, Piso 2   
3     N-1003 2023-05-10 03:00:00      Usuario_3   Paseo Gracia, 94, Piso 7   
4     N-1004 2023-02-25 17:00:00      Usuario_4    Calle Mayor, 80, Piso 8   
5     N-1005 2023-06-26 06:00:00      Usuario_5  Av. Libertad, 54, Piso 10   
6     N-1006 2023-04-18 20:00:00      Usuario_6   Av. Libertad, 48, Piso 7   
7     N-1007 2023-05-11 00:00:00      Usuario_7   Av. Libertad, 20, Piso 8   
8     N-1008 2023-05-21 08:00:00      Usuario_8   Av. Libertad, 73, Piso 6   
9     N-1009 2023-01-21 07:00:00      Usuario_9    Calle Mayor, 92, Piso 9   
10    N-1010 2023-03-21 15:00:00     Usuario_10    Calle Mayor, 61, Piso 6   
11    N-1011 2023-01-28 05:00:00     Usuario_11        Gran Vía,

In [13]:
print(df_norte.dtypes)

ID_Pedido                    object
Fecha_Transaccion    datetime64[ns]
Cliente_Nombre               object
Direccion_Envio              object
Producto                     object
Unidades                     object
Precio_Unitario              object
Total_Factura               float64
dtype: object


### **2. Sur (``ventas_sur.xlsx``):**
- Archivo Excel con múltiples hojas (``Q1_2023``, ``Q2_2023``).
- Contiene columnas booleanas (``Es_Cliente_Corporativo``) y de estado (``Estado_Envio``) que deben conservarse.
- Debes calcular una columna ``Total`` que no existe explícitamente ( ``Precio_Base``* ``Cantidad`` * (1 - ``Descuento_Aplicado``)).

In [11]:
!pip install openpyxl
!pip install xlrd

In [21]:
df_sur = pd.read_excel(
    "ventas_sur_v2.xlsx",
    sheet_name=None
)

df_sur = pd.concat(df_sur.values(),ignore_index=True)
df_sur["Total"] = df_sur["Precio_Base"] * df_sur["Cantidad"] * (1 -df_sur["Descuento_Aplicado"])
df_sur

,Ref_Venta,Fecha_Alta,Articulo,Cantidad,Precio_Base,Descuento_Aplicado,Es_Cliente_Corporativo,Estado_Envio,Total
0,S-5000,2023-02-11 00:00:00,Webcam HD,9,425.12,0.20,False,Enviado,3060.864
1,S-5001,2023-02-03 12:00:00,Teclado Mecánico,8,599.60,0.00,True,Devuelto,4796.800
2,S-5002,2023-04-06 07:00:00,Mouse Ergonómico,6,971.36,0.00,False,Completado,5828.160
3,S-5003,2023-04-02 05:00:00,Monitor 4K,1,799.66,0.10,True,Pendiente,719.694
4,S-5004,2023-06-17 12:00:00,Mouse Ergonómico,2,619.99,0.05,True,Devuelto,1177.981
...,...,...,...,...,...,...,...,...,...
155,S-5075,2023-05-17 22:00:00,Monitor 4K,2,1150.01,0.10,False,Enviado,2070.018
156,S-5076,2023-01-29 13:00:00,Webcam HD,1,874.12,0.10,True,Devuelto,786.708
157,S-5077,2023-06-09 11:00:00,Mouse Ergonómico,1,145.92,0.00,True,Devuelto,145.920
158,S-5078,2023-02-07 13:00:00,Monitor 4K,2,1152.24,0.00,False,Enviado,2304.480


### **3. Este (``ventas_este.json``):**

- JSON de con varios niveles de anidamiento
- La información útil está sepultada bajo ``data -> payload -> transaccion``.
- Debes extraer:
- ID del registro.
    - Fecha.
    - Ciudad del comprador.
    - Nombre del producto.
    - Cantidad.
    - Precio de lista y el monto del IVA (que está en un subdiccionario).
- Debes descartar metadatos técnicos (``latency_ms``, ``source_system``).

El DataFrame final debe tener **exactamente** estas columnas estandarizadas:
- ``id_transaccion``
- ``fecha``
- ``region``
- ``producto``
- ``cantidad``
- ``total_venta``
- ``ciudad``

In [24]:
df_este = pd.read_json("ventas_este_v2.json")
print(df_este)

                                          metadata  \
0    {'source_system': 'API_v2', 'latency_ms': 34}   
1    {'source_system': 'API_v2', 'latency_ms': 45}   
2    {'source_system': 'API_v2', 'latency_ms': 94}   
3    {'source_system': 'API_v2', 'latency_ms': 66}   
4    {'source_system': 'API_v2', 'latency_ms': 30}   
..                                             ...   
115  {'source_system': 'API_v2', 'latency_ms': 60}   
116  {'source_system': 'API_v2', 'latency_ms': 21}   
117  {'source_system': 'API_v2', 'latency_ms': 38}   
118  {'source_system': 'API_v2', 'latency_ms': 46}   
119  {'source_system': 'API_v2', 'latency_ms': 40}   

                                                  data  
0    {'id_registro': 'E-8000', 'payload': {'fecha_e...  
1    {'id_registro': 'E-8001', 'payload': {'fecha_e...  
2    {'id_registro': 'E-8002', 'payload': {'fecha_e...  
3    {'id_registro': 'E-8003', 'payload': {'fecha_e...  
4    {'id_registro': 'E-8004', 'payload': {'fecha_e...  
..       

In [17]:
import json
import pandas as pd

with open("ventas_este_v2.json") as f:
    data = json.load(f)

df_este = pd.json_normalize(data)


df_este = df_este[
    [
        "data.id_registro",
        "data.payload.fecha_evento",
        "data.payload.comprador.ubicacion.ciudad",
        "data.payload.transaccion.detalles_producto.nombre_comercial",
        "data.payload.transaccion.cantidad_comprada",
        "data.payload.transaccion.detalles_producto.precio_lista",
        "data.payload.transaccion.detalles_producto.impuestos.monto_iva"
    ]
]

df_este = df_este.rename(columns={
    "data.id_registro": "id_Transaccion",
    "data.payload.fecha_evento": "Fecha",
    "data.payload.comprador.ubicacion.ciudad": "Ciudad",
    "data.payload.transaccion.detalles_producto.nombre_comercial": "Producto",
    "data.payload.transaccion.cantidad_comprada": "Cantidad",
    "data.payload.transaccion.detalles_producto.precio_lista": "Total_Venta",
    "data.payload.transaccion.detalles_producto.impuestos.monto_iva": "Iva"
})

df.head(10)

,id_Transaccion,Fecha,Ciudad,Producto,Cantidad,Total_Venta,Iva
0,E-8000,2023-06-29 00:00:00,Sevilla,Monitor 4K,1,503,105.63
1,E-8001,2023-04-12 00:00:00,Sevilla,Webcam HD,1,1122,235.62
2,E-8002,2023-04-20 00:00:00,Sevilla,Webcam HD,2,994,208.74
3,E-8003,2023-04-14 00:00:00,Madrid,Webcam HD,2,1255,263.55
4,E-8004,2023-01-03 00:00:00,Sevilla,Monitor 4K,2,1458,306.18
5,E-8005,2023-02-24 00:00:00,Valencia,Teclado Mecánico,2,475,99.75
6,E-8006,2023-05-14 00:00:00,Sevilla,Mouse Ergonómico,2,498,104.58
7,E-8007,2023-02-22 00:00:00,Valencia,Webcam HD,2,1255,263.55
8,E-8008,2023-03-27 00:00:00,Madrid,Laptop Gamer,2,668,140.28
9,E-8009,2023-03-22 00:00:00,Sevilla,Docking Station,1,933,195.93
